## Decoding Customer Value: A SQL-Driven Retention Strategy

# Data Preparation

In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv("Dataset.csv")

# Find numerical columns
num_cols = df.select_dtypes(include="number").columns

# Fill missing values in each numerical column with that column's median
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Check if any nulls are left
print(df.isnull().sum())

# Save the cleaned file
df.to_csv("Dataset_cleaned.csv", index=False)

Customer ID               0
Age                       0
Gender                    0
Item Purchased            0
Category                  0
Purchase Amount (USD)     0
Location                  0
Size                      0
Color                     0
Season                    0
Review Rating             0
Subscription Status       0
Shipping Type             0
Discount Applied          0
Promo Code Used           0
Previous Purchases        0
Payment Method            0
Frequency of Purchases    0
dtype: int64


determining the metrics

In [2]:
cat_cols = df.columns

for col in cat_cols:
    print(f"\n{'='*50}")
    print(f"Column: {col}")
    print(f"Number of unique values: {df[col].nunique()}")
    print(df[col].unique())


Column: Customer ID
Number of unique values: 3900
[   1    2    3 ... 3898 3899 3900]

Column: Age
Number of unique values: 53
[55 19 50 21 45 46 63 27 26 57 53 30 61 65 64 25 52 66 31 56 18 38 54 33
 36 35 29 70 69 67 20 39 42 68 49 59 47 40 41 48 22 24 44 37 58 32 62 51
 28 43 34 23 60]

Column: Gender
Number of unique values: 2
['Male' 'Female']

Column: Item Purchased
Number of unique values: 25
['Blouse' 'Sweater' 'Jeans' 'Sandals' 'Sneakers' 'Shirt' 'Shorts' 'Coat'
 'Handbag' 'Shoes' 'Dress' 'Skirt' 'Sunglasses' 'Pants' 'Jacket' 'Hoodie'
 'Jewelry' 'T-shirt' 'Scarf' 'Hat' 'Socks' 'Backpack' 'Belt' 'Boots'
 'Gloves']

Column: Category
Number of unique values: 4
['Clothing' 'Footwear' 'Outerwear' 'Accessories']

Column: Purchase Amount (USD)
Number of unique values: 81
[ 53  64  73  90  49  20  85  34  97  31  68  72  51  81  36  38  48  62
  37  88  22  25  56  94  79  67  91  33  69  45  60  76  39 100  21  43
  54  55  30  28  59  24  83  46  70  29  26  58  96  40  32  95  41 

In [13]:
#────────────────────────────────────────────────────────────────
# FEATURE 1 : promo_dependency_score
# ────────────────────────────────────────────────────────────────
# Business question it answers:
#   "Is this customer buying because they love the brand, or only
#    because they were offered a discount?"
#
# Construction:
#   Both 'Discount Applied' and 'Promo Code Used' always move together in
#   this dataset (same 1,677 rows are Yes for both). So both flags
#   are summed and normalised to [0, 1].
#       0.0 = paid full price, no code used      (brand-driven buyer)
#       1.0 = discount applied AND promo code used (discount-driven buyer)
#
# Why this matters for the project:
#   The promotional sunset plan requires identifying WHICH segments to
#   stop discounting. A customer with score = 1.0 is the prime candidate.
#   But if they also have high previous purchases (Deal Hunter segment),
#   removing discounts abruptly risks losing real revenue — the sunset
#   must be gradual. This score makes that decision traceable.

df["promo_dependency_score"] = (df["Discount Applied"].map({'Yes': 1, 'No': 0}) + df["Promo Code Used"].map({'Yes': 1, 'No': 0})) / 2

print()
print("  [F1] promo_dependency_score")
vc = df["promo_dependency_score"].value_counts().sort_index()
print(f"       0.0 (no promo)   : {vc.get(0.0, 0):>5} customers ({vc.get(0.0, 0)/len(df):.1%})")
print(f"       1.0 (full promo) : {vc.get(1.0, 0):>5} customers ({vc.get(1.0, 0)/len(df):.1%})")

# ────────────────────────────────────────────────────────────────
# FEATURE 2 :total_spend
# ────────────────────────────────────────────────────────────────
# Business question it answers:
#   "How much revenue has this customer likely generated for the brand?"
#
# Construction:
#   (Previous Purchases + 1) × Purchase Amount (USD)
#   The +1 accounts for the current transaction in the dataset.
#
#   we use the best available proxy: total transactions × spend per
#   transaction. This assumes each historical purchase cost roughly the
#   same as the current one — reasonable given the narrow $20–$100 range
#   (std dev = $23.7, a tight distribution).
#
# Limitation we will state openly:
#   This overstates Customer's lifetime value for customers whose current purchase is unusually
#   high or low. But it is internally consistent and correlates with
#   segment behaviour, which is what the project requires.

df["total_spend"] = (df["Previous Purchases"] + 1) * df["Purchase Amount (USD)"]

print()
print("  [F2] total_spend (USD)")
print(f"       min    : ${df['total_spend'].min():.0f}")
print(f"       median : ${df['total_spend'].median():.0f}")
print(f"       mean   : ${df['total_spend'].mean():.0f}")
print(f"       max    : ${df['total_spend'].max():.0f}")

# ────────────────────────────────────────────────────────────────
# FEATURE 3 : satisfaction_flag
# ────────────────────────────────────────────────────────────────
# Business Question:
# "Is the customer satisfied with the brand experience?"
#
# Logic:
# Review Rating >= 4  → Satisfied
# Review Rating < 4   → Not Satisfied
#
# Why:
# High-value customers with low satisfaction are retention risks.
# High-value customers with high satisfaction are ideal customers.

df["satisfaction_flag"] = (
    df["Review Rating"] >= 4
).astype(int)

print()
print("  [F3] satisfaction_flag")
print(df["satisfaction_flag"].value_counts())

# ────────────────────────────────────────────────────────────────
# FEATURE 4 : purchase_frequency_score
# ────────────────────────────────────────────────────────────────
# Business question it answers:
#   "How frequently does this customer interact with the brand?"
#
# Construction:
#   The original column contains categorical labels such as:
#       Weekly, Monthly, Quarterly, Annually
#
#   These are converted into approximate purchases per year:
#
#       Weekly          → 52
#       Bi-Weekly       → 26
#       Fortnightly     → 24
#       Monthly         → 12
#       Quarterly       → 4
#       Every 3 Months  → 4
#       Annually        → 1
#
# Why this transformation:
#   The original text labels cannot be used directly in arithmetic,
#   correlation analysis, PCA, or SQL aggregations.
#
#   Converting them into yearly purchase estimates preserves the
#   relative intensity of purchasing behaviour and creates a metric
#   that can be compared across customers.
#
# Why it matters:
#   Customers who purchase more frequently are generally more engaged
#   with the brand and more likely to remain active in the future.
#
# Limitation:
#   These are approximate annual frequencies and not exact purchase
#   counts. They are intended to capture behavioural intensity rather
#   than precise transaction volume.

freq_map = {
    "Weekly": 52,
    "Bi-Weekly": 26,
    "Fortnightly": 24,
    "Monthly": 12,
    "Quarterly": 4,
    "Every 3 Months": 4,
    "Annually": 1
}

df["purchase_frequency_score"] = (
    df["Frequency of Purchases"]
    .map(freq_map)
)

print()
print("  [F4] purchase_frequency_score")

for label, score in sorted(freq_map.items(),
                           key=lambda x: -x[1]):
    count = (
        df["Frequency of Purchases"]
        == label
    ).sum()

    print(
        f"       {label:<20}"
        f" → {score:>3}"
        f" ({count} customers)"
    )

# ────────────────────────────────────────────────────────────────
# FEATURE 5 : engagement_score
# ────────────────────────────────────────────────────────────────
# Business question it answers:
#   "How actively engaged is this customer with the brand?"
#
# Why this is different from total_spend:
#   Spending measures monetary value, but engagement measures
#   behavioural commitment.
#
#   A customer may not spend the most money per purchase, yet may
#   interact frequently, subscribe to updates, and repeatedly
#   return to the brand. Such customers often have strong future
#   retention potential.
#
# Construction:
#   Composite score built from:
#
#       purchase_frequency_score
#       Previous Purchases
#       Subscription Status
#
#   All continuous variables are min-max normalised to [0,1]
#   before combination.
#
# Why min-max normalisation:
#   purchase_frequency_score ranges from 1–52 while Previous
#   Purchases ranges from 0–50.
#
#   Without normalisation, larger-scale variables would dominate
#   the final score.
#
# Why equal weights:
#   The dataset contains no churn labels or future behaviour data
#   that would justify assigning different weights.
#
#   Equal weighting provides a simple and transparent measure of
#   overall engagement.
#
# Why it matters:
#   Engagement becomes one of the inputs for Value Tier creation,
#   helping distinguish genuinely involved customers from those
#   who simply made a few expensive purchases.

def min_max(series):
    return (
        (series - series.min())
        /
        (series.max() - series.min())
    )

df["subscription_flag"] = (
    df["Subscription Status"]
    .map({"Yes": 1, "No": 0})
)

df["engagement_score"] = (
    min_max(df["purchase_frequency_score"]) * (1/3)
    +
    min_max(df["Previous Purchases"]) * (1/3)
    +
    df["subscription_flag"] * (1/3)
)

print()
print("  [F5] engagement_score")

print(
    f"Min: {df['engagement_score'].min():.3f}"
)
print(
    f"Median: {df['engagement_score'].median():.3f}"
)
print(
    f"Mean: {df['engagement_score'].mean():.3f}"
)
print(
    f"Max: {df['engagement_score'].max():.3f}"
)





  [F1] promo_dependency_score
       0.0 (no promo)   :  2223 customers (57.0%)
       1.0 (full promo) :  1677 customers (43.0%)

  [F2] total_spend (USD)
       min    : $42
       median : $1330
       mean   : $1578
       max    : $5100

  [F3] satisfaction_flag
satisfaction_flag
0    2266
1    1634
Name: count, dtype: int64

  [F4] purchase_frequency_score
       Weekly               →  52 (539 customers)
       Bi-Weekly            →  26 (547 customers)
       Fortnightly          →  24 (542 customers)
       Monthly              →  12 (553 customers)
       Quarterly            →   4 (563 customers)
       Every 3 Months       →   4 (584 customers)
       Annually             →   1 (572 customers)

  [F5] engagement_score
Min: 0.000
Median: 0.333
Mean: 0.361
Max: 1.000


In [14]:
# ────────────────────────────────────────────────────────────────
# FEATURE 6 : value_score (PCA Based)
# ────────────────────────────────────────────────────────────────
# Business question it answers:
#   "How valuable is this customer to the business when both
#    monetary and behavioural factors are considered?"
#
# Construction:
#   Principal Component Analysis (PCA) is applied on:
#
#       total_spend
#       engagement_score
#       promo_dependency_score
#       satisfaction_flag
#
# Why PCA:
#   The dataset does not contain a predefined customer value
#   metric. Assigning manual weights (for example 40% spend,
#   30% engagement, etc.) would be subjective.
#
#   PCA provides a data-driven approach by learning the
#   combination of variables that explains the greatest
#   variation across customers.
#
# Why these variables:
#
#   total_spend
#       → monetary value
#
#   engagement_score
#       → behavioural commitment
#
#   satisfaction_flag
#       → customer experience
#
#   promo_dependency_score
#       → discount reliance
#
# Together these variables capture both current value and
# future retention potential.
#
# Why it matters:
#   The resulting score becomes the foundation for customer
#   prioritisation and value-based segmentation.

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

value_features = [
    "total_spend",
    "engagement_score",
    "promo_dependency_score",
    "satisfaction_flag"
]

scaler = StandardScaler()

X_scaled = scaler.fit_transform(
    df[value_features]
)

pca = PCA(n_components=1)

df["value_score"] = pca.fit_transform(
    X_scaled
)

# Ensure higher score means higher value

if (
    df["value_score"]
    .corr(df["total_spend"])
    < 0
):
    df["value_score"] *= -1

print()
print("  [F6] value_score")

print(
    "       Variance Explained:",
    round(
        pca.explained_variance_ratio_[0],
        3
    )
)

weights = pd.DataFrame({
    "Feature": value_features,
    "Weight": pca.components_[0]
})

print()
print("       PCA Weights")
print(weights)


  [F6] value_score
       Variance Explained: 0.407

       PCA Weights
                  Feature    Weight
0             total_spend  0.423255
1        engagement_score  0.705278
2  promo_dependency_score  0.568644
3       satisfaction_flag  0.009060


In [15]:
# ────────────────────────────────────────────────────────────────
# FEATURE 7 : value_tier
# ────────────────────────────────────────────────────────────────
# Business question it answers:
#   "Which customers should receive the greatest retention
#    investment and attention?"
#
# Construction:
#   Customers are ranked according to value_score and divided
#   into three equal groups:
#
#       Bottom 33%  → Low Value
#       Middle 33%  → Mid Value
#       Top 33%     → High Value
#
# Why percentile segmentation:
#   Fixed thresholds are arbitrary and require external
#   benchmarks.
#
#   Percentile-based segmentation automatically adapts to
#   the customer distribution and creates stable groups.
#
# Why it matters:
#   Value Tier becomes the primary variable used throughout
#   SQL analysis, dashboarding, and retention strategy.

# Convert PCA score to 0–100 scale

df["value_score"] = (
    (
        df["value_score"]
        -
        df["value_score"].min()
    )
    /
    (
        df["value_score"].max()
        -
        df["value_score"].min()
    )
) * 100

df["value_tier"] = pd.qcut(
    df["value_score"],
    q=3,
    labels=[
        "Low Value",
        "Mid Value",
        "High Value"
    ]
)

print()
print("  [F7] value_tier")

print(
    df["value_tier"]
    .value_counts()
)


  [F7] value_tier
value_tier
Low Value     1300
Mid Value     1300
High Value    1300
Name: count, dtype: int64


In [17]:
# ────────────────────────────────────────────────────────────────
# FEATURE 8 : loyalty_def_A
# ────────────────────────────────────────────────────────────────
# Business question it answers:
#   "Who are genuinely loyal customers rather than customers
#    who purchase only because of discounts?"
#
# Construction:
#   Loyal = Previous Purchases > dataset median
#           AND promo_dependency_score = 0
#
# Why:
#   A loyal customer should demonstrate repeat purchase
#   behaviour while being independent of promotions.

median_prev = df["Previous Purchases"].median()

df["loyalty_def_A"] = (
    (
        df["Previous Purchases"] > median_prev
    )
    &
    (
        df["promo_dependency_score"] == 0
    )
).astype(int)

print()
print("  [F8] loyalty_def_A")

n_loyal = df["loyalty_def_A"].sum()

print(
    f"       Loyal Customers : {n_loyal}"
)

print(
    f"       Percentage      : {n_loyal/len(df):.1%}"
)


# ────────────────────────────────────────────────────────────────
# FEATURE 9 : loyalty_def_B
# ────────────────────────────────────────────────────────────────
# Business question it answers:
#   "Does formal engagement with the brand correspond
#    to customer loyalty?"
#
# Construction:
#   Loyal = Subscribed
#           AND Purchase Frequency >= Monthly
#
# Why:
#   Subscription represents explicit brand engagement.
#   Monthly-or-higher frequency ensures active participation.

df["loyalty_def_B"] = (
    (
        df["subscription_flag"] == 1
    )
    &
    (
        df["purchase_frequency_score"] >= 12
    )
).astype(int)

print()
print("  [F9] loyalty_def_B")
n_loyal = df["loyalty_def_B"].sum()
print(
    f"       Loyal Customers : {n_loyal}"
)
print(
    f"       Percentage      : {n_loyal/len(df):.1%}"
)


  [F8] loyalty_def_A
       Loyal Customers : 1103
       Percentage      : 28.3%

  [F9] loyalty_def_B
       Loyal Customers : 599
       Percentage      : 15.4%


In [18]:
# ────────────────────────────────────────────────────────────────
# FEATURE 10 : customer_segment
# ────────────────────────────────────────────────────────────────
# Business question it answers:
#   "What actionable retention segment does this
#    customer belong to?"
#
# Segments:
#   Champion
#   Loyalist
#   Deal Hunter
#   At Risk
#   Potential Loyalist
#   Passive

def assign_segment(row):

    loyal = row["loyalty_def_A"]
    promo = row["promo_dependency_score"]
    tier = row["value_tier"]
    freq = row["purchase_frequency_score"]

    if loyal == 1 and tier == "High Value":
        return "Champion"

    elif loyal == 1:
        return "Loyalist"

    elif promo == 1.0 and tier == "Low Value":
        return "At Risk"

    elif promo == 1.0:
        return "Deal Hunter"

    elif promo == 0 and freq >= 12:
        return "Potential Loyalist"

    else:
        return "Passive"

df["customer_segment"] = (
    df.apply(assign_segment, axis=1)
)

print()
print("  [F10] customer_segment")

print(
    df["customer_segment"]
    .value_counts()
)


  [F10] customer_segment
customer_segment
Deal Hunter           1618
Loyalist               973
Potential Loyalist     632
Passive                488
Champion               130
At Risk                 59
Name: count, dtype: int64


In [19]:
# ────────────────────────────────────────────────────────────────
# FINAL OUTPUT
# ────────────────────────────────────────────────────────────────

df.to_csv(
    "enriched_dataset.csv",
    index=False
)

print()
print("Final dataset saved successfully.")
print(
    f"Rows    : {df.shape[0]}"
)
print(
    f"Columns : {df.shape[1]}"
)


Final dataset saved successfully.
Rows    : 3900
Columns : 31
